# ECC/EMC — Kaggle Training: FI Dataset

Supervised (Table 4) + Semi-supervised (Section 4.2.1) on the FI dataset.

## Before you run
1. **Enable GPU:** Settings → Session options → GPU (P100 or T4). Phone verification required.
2. **Add datasets:** Add your private FI dataset (resized 256×256) in the notebook sidebar.
3. **Add GitHub token secret:** Kaggle Settings → Secrets → `GITHUB_TOKEN` = your GitHub PAT (repo scope).
4. **Edit `FI_DATASET_SLUG`** below with your private FI dataset slug.

## Results
All outputs go to `/kaggle/working/runs/fi_*` — separate from EmoSet results.

In [ ]:
# ===================== CONFIGURE THESE =====================
# Your private FI dataset (resized 256x256, made with scripts/resize_and_split.py)
FI_DATASET_SLUG = "your-username/fi-dataset-256"   # <-- REPLACE

# Which experiments to run
RUN_SUPERVISED = True
RUN_SSL = True
# ==========================================================

print("FI dataset slug:", FI_DATASET_SLUG)

In [ ]:
# Install dependencies
!pip -q install torch torchvision numpy pyyaml pillow

In [ ]:
# Clone repo using GitHub token secret (added in Kaggle Settings → Secrets)
import os
from ecc.kaggle_utils import clone_with_token

repo_url = "https://github.com/Rahul-Sai-Indeevar/To-Err-Like-Human.git"
dest = "/kaggle/working/ecc-ver"

if not os.path.exists(dest):
    clone_with_token(repo_url, dest)

os.chdir(dest)
print("Cloned repo to", dest)
print("Current dir:", os.getcwd())

In [ ]:
# Install ecc-ver package
!pip -q install -e .

In [ ]:
# Symlink FI dataset into data/ layout
import os

os.makedirs("data", exist_ok=True)

# Find FI/ and FI_splits/ inside the mounted private dataset
fi_input = f"/kaggle/input/{FI_DATASET_SLUG.split('/')[1]}" if "/" in FI_DATASET_SLUG else f"/kaggle/input/{FI_DATASET_SLUG}"
print("Looking for FI at:", fi_input)

found = False
for root, dirs, files in os.walk(fi_input):
    for d in dirs:
        if d in ("FI", "FI_splits"):
            src = os.path.join(root, d)
            dst = f"data/{d}"
            if not os.path.exists(dst):
                os.symlink(src, dst)
                print(f"Linked {d} -> {dst}")
            found = True

if not found:
    print("WARNING: Could not find FI/ or FI_splits/ under", fi_input)

print("\nData layout:")
!find data -maxdepth 3 | head -30

In [ ]:
# Quick smoke test (CPU, no data needed)
!python -m pytest -q

In [ ]:
# ===================== SUPERVISED (FI, ResNet50, Table 4) =====================
if RUN_SUPERVISED:
    # alpha = 0 (pure CE)
    print("=== alpha=0 ===")
    !python -m ecc.supervised --config configs/fi_resnet50.yaml --set loss.alpha=0.0 output_dir=/kaggle/working/runs/fi_r50_a0

    # alpha = 0.2
    print("=== alpha=0.2 ===")
    !python -m ecc.supervised --config configs/fi_resnet50.yaml --set loss.alpha=0.2 output_dir=/kaggle/working/runs/fi_r50_a0.2

    # alpha = 1.0
    print("=== alpha=1.0 ===")
    !python -m ecc.supervised --config configs/fi_resnet50.yaml --set loss.alpha=1.0 output_dir=/kaggle/working/runs/fi_r50_a1.0

print("Supervised done.")

In [ ]:
# ===================== SEMI-SUPERVISED (FI, Section 4.2.1) =====================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if RUN_SSL:
    # 80 labels, EMC-adaptive threshold
    print("=== SSL n=80, rule=ratio (EMC-adaptive) ===")
    !python -m ecc.ssl --config configs/ssl_fi.yaml --set ssl.n_labeled=80 ssl.rule=ratio output_dir=/kaggle/working/runs/fi_ssl_n80_ratio

    # 800 labels, EMC-adaptive threshold
    print("=== SSL n=800, rule=ratio (EMC-adaptive) ===")
    !python -m ecc.ssl --config configs/ssl_fi.yaml --set ssl.n_labeled=800 ssl.rule=ratio output_dir=/kaggle/working/runs/fi_ssl_n800_ratio

    # 1600 labels, EMC-adaptive threshold
    print("=== SSL n=1600, rule=ratio (EMC-adaptive) ===")
    !python -m ecc.ssl --config configs/ssl_fi.yaml --set ssl.n_labeled=1600 ssl.rule=ratio output_dir=/kaggle/working/runs/fi_ssl_n1600_ratio

    # 800 labels, fixed threshold (FixMatch baseline)
    print("=== SSL n=800, rule=fixed (FixMatch baseline) ===")
    !python -m ecc.ssl --config configs/ssl_fi.yaml --set ssl.n_labeled=800 ssl.rule=fixed output_dir=/kaggle/working/runs/fi_ssl_n800_fixed

print("SSL done.")

In [ ]:
# ===================== PREPARE RESULTS ZIP FOR ONE-CLICK DOWNLOAD =====================
import zipfile, os, glob, shutil

runs_dir = "/kaggle/working/runs"
zip_path = "/kaggle/working/fi_results.zip"

run_dirs = sorted(glob.glob(os.path.join(runs_dir, "fi_*")))

if not run_dirs:
    print("No fi_* run directories found under", runs_dir)
else:
    stage_root = os.path.join(runs_dir, "_zip_stage")
    os.makedirs(stage_root, exist_ok=True)
    
    for run_dir in run_dirs:
        run_name = os.path.basename(run_dir)
        stage = os.path.join(stage_root, run_name)
        os.makedirs(stage, exist_ok=True)
        
        # best.pt -> best_model.pt
        best_pt = os.path.join(run_dir, "best.pt")
        results_json = os.path.join(run_dir, "results.json")
        history_jsonl = os.path.join(run_dir, "history.jsonl")
        config_json = os.path.join(run_dir, "config.json")
        
        if os.path.exists(best_pt):
            shutil.copy2(best_pt, os.path.join(stage, "best_model.pt"))
        if os.path.exists(results_json):
            shutil.copy2(results_json, os.path.join(stage, "results.json"))
        if os.path.exists(history_jsonl):
            shutil.copy2(history_jsonl, os.path.join(stage, "history.jsonl"))
        if os.path.exists(config_json):
            shutil.copy2(config_json, os.path.join(stage, "config.json"))
        
        print(f"Staged {run_name}")
    
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for stage in sorted(glob.glob(os.path.join(stage_root, "fi_*"))):
            run_name = os.path.basename(stage)
            for fname in sorted(os.listdir(stage)):
                fpath = os.path.join(stage, fname)
                zf.write(fpath, arcname=os.path.join(run_name, fname))
    
    shutil.rmtree(stage_root, ignore_errors=True)
    
    print(f"\nCreated {zip_path}")
    print("Contents:")
    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            print(f"  {info.filename} ({info.file_size:,} bytes)")
    
    print("\nDownload fi_results.zip from the notebook output panel (right sidebar → Output).")

## Resuming after a session timeout

Kaggle free sessions are capped at ~12 hours. Every epoch saves `checkpoint.pt` in the run's
`output_dir`. If a session dies, re-run the same cell — it auto-detects `checkpoint.pt`
and resumes from the last epoch.

To force a specific checkpoint, add `--set resume_from=/kaggle/working/runs/<run>/checkpoint.pt`.

## Download results

Run the cell above to create `fi_results.zip` containing, for each run:
- `best_model.pt` — best-on-val model weights (from `best.pt`)
- `results.json` — test metrics (ACC, ECC, EMC, ACC2)
- `history.jsonl` — per-epoch training log
- `config.json` — exact config used

Download `fi_results.zip` from the notebook output panel (right sidebar → Output) before
the session ends.